# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = "task281"
CH = 10
H = W = 30
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
LOCAL_TASK_JSON = Path("/mnt/data/task281.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / "task281.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = WORK_DIR / "task281_marker_guided_frame_expansion_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = WORK_DIR / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
with TASK_JSON.open("r") as f:
    task = json.load(f)
print(TASK_ID, len(task.get("train", [])), len(task.get("test", [])), len(task.get("arc-gen", [])))

task281 3 1 262


In [6]:
def grid_to_tensor(grid, h=H, w=W, ch=CH):
    """Encode an ARC grid as [1,10,30,30].

    Grid cells, including color-0 background inside the true grid, are one-hot.
    Padding outside the true grid is all-zero across channels, so active_canvas is
    exactly sum(input_channels) > 0.
    """
    gh, gw = len(grid), len(grid[0])
    assert gh <= h and gw <= w, f"grid too large for static ONNX contract: {gh}x{gw}"
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            x[0,int(v),r,c]=1.0
    return x

class Task281MarkerGuidedFrameExpansion(nn.Module):
    """Static symbolic model for marker-guided frame expansion.

    Rule:
      1. active_canvas = sum(input_channels) > 0.
      2. color 8 is the expansion marker and is removed from the output.
      3. source object = nonzero colors excluding marker 8.
      4. compute the source object's rectangular bounding box.
      5. expand that box just enough to include the marker coordinate.
      6. infer border color from the original frame boundary and interior color
         from the original frame interior, then redraw the expanded frame.
      7. force outside active_canvas to all-zero across all channels.

    The implementation uses reductions and static 30x30 coordinate fields;
    no Loop, Scan, NonZero, Unique, Script, or Function ops are required.
    """
    def __init__(self):
        super().__init__()
        rr=torch.arange(H,dtype=torch.float32).view(1,1,H,1).expand(1,1,H,W)
        cc=torch.arange(W,dtype=torch.float32).view(1,1,1,W).expand(1,1,H,W)
        self.register_buffer("rr", rr)
        self.register_buffer("cc", cc)

    def forward(self, x):
        active=(x.sum(dim=1, keepdim=True)>0.5).float()
        marker=x[:,8:9]*active

        # Source frame support: foreground colors excluding the color-8 marker.
        non_bg=x[:,1:10].sum(dim=1, keepdim=True)
        obj=((non_bg-marker)>0.5).float()*active

        # Source frame bounding box, computed without NonZero.
        row_presence=(obj.amax(dim=3)>0.5).float()
        col_presence=(obj.amax(dim=2)>0.5).float()
        min_r=torch.argmax(row_presence, dim=2).to(torch.float32).view(1,1,1,1)
        max_r=(float(H-1)-torch.argmax(torch.flip(row_presence,dims=[2]), dim=2).to(torch.float32)).view(1,1,1,1)
        min_c=torch.argmax(col_presence, dim=2).to(torch.float32).view(1,1,1,1)
        max_c=(float(W-1)-torch.argmax(torch.flip(col_presence,dims=[2]), dim=2).to(torch.float32)).view(1,1,1,1)

        # Marker coordinate.
        mflat=marker.reshape(1,-1)
        midx=torch.argmax(mflat, dim=1).to(torch.float32).view(1,1,1,1)
        mr=torch.floor(midx/float(W))
        mc=midx-mr*float(W)

        # New frame bounds are the source bounds expanded to include marker.
        new_min_r=torch.minimum(min_r, mr)
        new_max_r=torch.maximum(max_r, mr)
        new_min_c=torch.minimum(min_c, mc)
        new_max_c=torch.maximum(max_c, mc)

        old_bbox=((self.rr>=min_r)&(self.rr<=max_r)&(self.cc>=min_c)&(self.cc<=max_c)).float()
        old_boundary=old_bbox*(((self.rr==min_r)|(self.rr==max_r)|(self.cc==min_c)|(self.cc==max_c)).float())
        old_inner=old_bbox*(1.0-old_boundary)

        # Per-channel semantic gates: border color from old boundary, interior color from old interior.
        border_gate=((x*old_boundary).sum(dim=(2,3), keepdim=True)>0.5).float()
        inner_gate=((x*old_inner).sum(dim=(2,3), keepdim=True)>0.5).float()

        # Exclude background channel and the marker channel from drawable colors.
        channel_ids=torch.arange(CH,dtype=torch.float32,device=x.device).view(1,CH,1,1)
        nonzero_nonmarker=(((channel_ids>0.5)&(channel_ids<7.5)) | (channel_ids>8.5)).float()
        border_gate=border_gate*nonzero_nonmarker
        inner_gate=inner_gate*nonzero_nonmarker

        new_bbox=((self.rr>=new_min_r)&(self.rr<=new_max_r)&(self.cc>=new_min_c)&(self.cc<=new_max_c)).float()*active
        new_boundary=new_bbox*(((self.rr==new_min_r)|(self.rr==new_max_r)|(self.cc==new_min_c)|(self.cc==new_max_c)).float())
        new_inner=new_bbox*(1.0-new_boundary)

        fg=(new_boundary*border_gate + new_inner*inner_gate)*active
        occ=torch.clamp(fg[:,1:10].sum(dim=1, keepdim=True),0.0,1.0)
        bg=active*(1.0-occ)
        return torch.cat([bg,fg[:,1:10]], dim=1)*active

model = Task281MarkerGuidedFrameExpansion().eval()

# Symbolic sanity check before export.
with torch.no_grad():
    for split in ["train", "test", "arc-gen"]:
        ok=0; bad=[]
        for i,ex in enumerate(task.get(split, [])):
            y=model(torch.from_numpy(grid_to_tensor(ex["input"]))).numpy()
            exp=grid_to_tensor(ex["output"])
            if np.array_equal((y>0.5).astype(np.float32), exp):
                ok += 1
            else:
                bad.append(i)
        print(split, ok, "/", len(task.get(split, [])), "bad", bad[:10])

train 3 / 3 bad []
test 1 / 1 bad []
arc-gen 262 / 262 bad []


In [7]:
dummy = torch.from_numpy(grid_to_tensor(task["test"][0]["input"]))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=["input"], output_names=["output"],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
ONNX_PATH, ONNX_PATH.stat().st_size

/tmp/ipykernel_16/1648413716.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


(PosixPath('/kaggle/working/task281_marker_guided_frame_expansion_onnx/task281.onnx'),
 25375)

In [8]:
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
summary_static = {
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "function_count": len(onnx_model.functions),
}
print(summary_static)
assert summary_static["input_shape"] == [1,10,30,30]
assert summary_static["output_shape"] == [1,10,30,30]
assert summary_static["onnx_size_bytes"] < 1_400_000
assert not summary_static["forbidden_ops"]
assert summary_static["function_count"] == 0

{'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30], 'onnx_size_bytes': 25375, 'ops': {'Constant': 45, 'ReduceSum': 5, 'Greater': 6, 'Cast': 15, 'Slice': 5, 'Mul': 17, 'Sub': 7, 'ReduceMax': 2, 'ArgMax': 5, 'Reshape': 6, 'Div': 1, 'Floor': 1, 'Min': 2, 'Max': 2, 'GreaterOrEqual': 4, 'LessOrEqual': 4, 'And': 6, 'Equal': 8, 'Or': 6, 'Add': 1, 'Clip': 1, 'Concat': 1}, 'forbidden_ops': [], 'function_count': 0}


In [9]:
sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=["CPUExecutionProvider"])

def validate_examples(examples):
    ok=0; bad=[]; outside_zero_ok=0; active_canvas_covered_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex["input"])
        y=sess.run(None,{"input":x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=grid_to_tensor(ex["output"])
        if np.array_equal(pred,exp):
            ok += 1
        else:
            bad.append(i)
        active=(x.sum(axis=1,keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
        active_canvas_covered_ok += bool(np.all((pred.sum(axis=1,keepdims=True)>0.5)==active))
    return {"ok":ok,"total":len(examples),"bad_first10":bad[:10],
            "outside_zero_ok":outside_zero_ok,
            "active_canvas_covered_ok":active_canvas_covered_ok}

rng=random.Random(0)
inds=list(range(len(task.get("arc-gen",[]))))
rng.shuffle(inds)
hold=[task["arc-gen"][i] for i in inds[:max(1, int(round(0.60*len(inds))))]] if inds else []
summary = {
    **summary_static,
    "train": validate_examples(task.get("train", [])),
    "test": validate_examples(task.get("test", [])),
    "arc_gen_60pct_holdout": validate_examples(hold),
    "arc_gen_all": validate_examples(task.get("arc-gen", [])),
}
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
assert summary["train"]["ok"] == summary["train"]["total"]
assert summary["test"]["ok"] == summary["test"]["total"]
assert summary["arc_gen_60pct_holdout"]["ok"] == summary["arc_gen_60pct_holdout"]["total"]
assert summary["arc_gen_all"]["ok"] == summary["arc_gen_all"]["total"]

{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 25375,
  "ops": {
    "Constant": 45,
    "ReduceSum": 5,
    "Greater": 6,
    "Cast": 15,
    "Slice": 5,
    "Mul": 17,
    "Sub": 7,
    "ReduceMax": 2,
    "ArgMax": 5,
    "Reshape": 6,
    "Div": 1,
    "Floor": 1,
    "Min": 2,
    "Max": 2,
    "GreaterOrEqual": 4,
    "LessOrEqual": 4,
    "And": 6,
    "Equal": 8,
    "Or": 6,
    "Add": 1,
    "Clip": 1,
    "Concat": 1
  },
  "forbidden_ops": [],
  "function_count": 0,
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first10": [],
    "outside_zero_ok": 3,
    "active_canvas_covered_ok": 3
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_canvas_covered_ok": 1
  },
  "arc_gen_60pct_holdout": {
    "ok": 157,
    "total": 157,
    "bad_first10": [],
    "outside_zero_ok": 157,
    "active_canvas_covered_ok": 157
  },
  "arc_gen_all": {


In [10]:

with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']


Wrote: /kaggle/working/submission.zip
Zip contents: ['task281.onnx']
